Una funzione hash prende un input $X$ di lunghezza variabile e produce un risultato di lunghezza fissa $Y = H(X)$. Il risultato è noto come **digest** o **impronta** (fingerprint) del messaggio. Ad es. SHA256 produce sempre un output di 256 bit (32 byte).

La funzione hash deve essere **efficiente da calcolare**. Le funzioni hash si usano anche fuori dalla crittografia, ad esempio nelle tabelle hash per l'indicizzazione. Ad esempio $H(\text{parola}) = \text{prima lettera della parola}$ è una funzione hash valida, ma è inadatta alla sicurezza. 

Una **funzione hash crittografica** deve soddisfare requisiti aggiuntivi. 

Immaginiamo di avere due testi quasi identici con solo una lettera diversa. Anzitutto una buona funzione hash crittografica dovrebbe produrre due digest completamente diversi e privo di schemi evidenti per questi due testi. In generale si vuole che **una piccola modifica dell'input, anche un solo bit, dovrebbe produrre un output con idealmente almeno la metà dei bit output diversi**. Questo principio è noto come **effetto valanga** (avalanche effect).

La funzione rimane comunque **deterministica**: lo stesso input produce sempre lo stesso output. Il secondo aspetto è che un attaccante non dovrebbe riuscire a modificare intenzionalmente un messaggio mantenendo lo stesso digest.  
Ad es. una funzione hash del tipo:
$$H(X) = \text{somma delle cifre di X} mod 10$$
non è sicura in questo senso perché basta prendere un X con una cifra aumentata di 1 e un'altra diminuita di 1 per ottenere lo stesso digest. 

Con "un attaccante non dovrebbe" intendiamo non che sia matematicamente impossibile, ma che sia **computazionalmente difficile**.  

### Prima proprietà: **Preimage resistance aka one-wayness**
**Dato un digest $Y$, deve essere computazionalmente difficile trovare un input $X$ tale che $H(X) = Y$.**

In simboli l'attaccante riceve $Y$ e deve trovare un $X$ tale che $H(X) = Y$.  
Importante capire la differenza tra non invertibilità e difficoltà di trovare una preimmagine. Una funzione hash è molti-a-uno, per cui diversi input possono produrre lo stesso digest --> il digest non identifica un unico messaggio originale. La proprietà richiesta è quindi più precisa e ben diversa dalla non invertibilità: deve essere **computazionalmente difficile** (e quindi non impossibile) trovare anche un solo input che produca un digest specifico.

Supponiamo l'hash abbia un digest di $n$ bit e si comporti come una funzione hash casuale. Allora gli output possibili sono $N = 2^n$. Per un digest bersaglio $Y$, un tentativo indipendente ha probabilità $p = \frac{1}{2^n}$ di produrre proprio $Y$. Il numero medio di tentativi necessari per il primo successo è quindi $E = \frac{1}{p} = 2^n$ (distribuzione geometrica).  

Con soli 16 bit i tentativi sono $2^{16} = 65536$, un numero piccolo per i computer moderni. Per questo un digest crittografico deve essere sufficientemente lungo. Tuttavia la lunghezza non basta: il ragionamento precedente si basa su una ricerca generica per tentativi. Se la struttura della funzione presenta debolezze, un attaccante potrebbe trovare metodi più efficienti.  
Ad esempio SHA1 produce un digest di 160 bit, ma è stato dimostrato come vulnerabile ad attacchi crittoanalitici (in particolare riguardo le **collisioni**)

### Seconda proprietà: **Second preimage resistance**
Detta anche **weak collision resistance**, richiede che **dato un input $X_1$, sia computazionalmente difficile trovare un secondo input $X_2$ tale che $H(X_1) = H(X_2)$**.

Il punto di questa proprietà è che in questo caso l'attaccante conosce già un input che produce il digest bersaglio -> potrebbe sfruttare questa informazione per costruirne un altro (es. proteggo un documento in H(X), l'avversario vuole produrre un documento diverso che produca lo stesso digest).

Per un hash ideale random con $n$ bit, il costo generico per trovare una seconda preimmagine è dell'ordine di $2^n$ tentativi (probabilità di beccare proprio il digest bersaglio $p = \frac{1}{2^n}$, e di nuovo assumendo tentativi indipendenti geometrica).

Vediamo con un esempio una funzione che può essere one-way ma avere collisioni facili:
$$F(x) = g^x \mod p$$
dove $p$ è un numero primo e $g$ valore opportunamente scelto non divisibile per $p$. Calcolare $F(x)$ conoscendo $x$ è facile ma il problema inverso di ricavare $x$ da $F(x)$ è difficile (problema del logaritmo discreto). 
Questo spiega la one-wayness.  
Tuttavia, se conosco già $x$, trovare un altro input con lo stesso risultato è facile. Per il piccolo teorema di Fermat:
$$g^{p-1} \equiv 1 \mod p$$
Quindi:
$$F(x+p-1) = g^{x+p-1} \mod p = g^x g^{p-1} \mod p = g^x \mod p = F(x)$$
basta quindi scegliere $x' = x + p - 1$ per ottenere un secondo input che produce lo stesso digest.

In soldoni la difficoltà di risalire dall'output a un input non garantisce la difficoltà di trovare un secondo input, quando il primo è noto.

### Terza proprietà: **Collision resistance**
Detta anche **strong collision resistance**, richiede che **sia computazionalmente difficile trovare due input distinti $X_1$ e $X_2$ tali che $H(X_1) = H(X_2)$**.

Questa volta non c'è un messaggio bersaglio fissato: l'attaccante può scegliere entrambi i messaggi --> gli basta trovare una qualunque coppia. Pertanto trovare una collisione generica è più facile che trovare una seconda preimmagine per un digest specifico --> una funzione hash crittografica deve resistere anche a questa maggiore libertà dell'attaccante.

Per capire il numero medio di tentativi bruteforce necessari all'attaccante per trovare una collisione, considerando sempre un hash ideale con $n$ bit, possiamo usare il **paradosso del compleanno**. 

Siamo 23 persone. Qual è la probabilità che nessuno faccia il compleanno il mio stesso giorno?
$$(\frac{364}{365})^{22} \approx 0.94$$
Invece, tra 23 persone qual è la probabilità almeno due di loro abbiano lo stesso compleanno? Assumendo 365 giorni equiprobabili e compleanni indipendenti, la probabilità che tutti abbiano compleanni diversi è:
$$\Pr(\text{tutti diversi}) = \frac{365}{365} \cdot \frac{364}{365} \cdot \frac{363}{365} \cdots \frac{343}{365} \approx 0.493$$
La probabilità di almeno una coincidenza è quindi:
$$1 - 0.493 \approx 0.507$$
La differenza deriva dal numero di confronti: nel primo caso confrontiamo tutti con un giorno fissato, nel secondo invece considero tutte le $\binom{23}{2} = 253$ coppie possibili.

Generalizziamo con la formula del **birthday bound**. Indichiamo con $n$ il numero di bit del digest, $N = 2^n$ il numero di possibili digest, e con $k$ il numero di messaggi hashati. Assumendo hash ideale e quindi output indipendenti e uniformi, la probabilità di nessuna collisione è:
$$\Pr(\text{nessuna collisione}) = \prod_{i=0}^{k-1} \frac{N-i}{N} = \prod_{i=0}^{k-1} (1 - \frac{i}{N})$$
(il primo digest può essere uno qualsiasi, il secondo deve evitare un valore, il terzo due e così via...).

Usando l'approssimazione $1-x \approx e^{-x}$ per $x$ piccolo, otteniamo:
$$\Pr(\text{nessuna collisione}) \approx \prod_{i=0}^{k-1} e^{-i/N} = e^{-\sum_{i=0}^{k-1} i/N} = e^{-k(k-1)/2N}$$
Per raggiungere una probabilità di collisione del $50\%$, dobbiamo risolvere:
$$\Pr(\text{nessuna collisione}) = e^{-k(k-1)/2N} = 0.5$$
e otteniamo:
$$k(k-1) \approx k^2 \approx 2N \ln 2 = 1.386 N$$
Per cui $k \approx \sqrt{1.386 N}$, e poiché $N = 2^n$, otteniamo $k \approx 1.177 \cdot 2^{n/2}$.

In sostanza un hash ideale con digest di $n$ bit richiede circa $2^{n/2}$ tentativi prima che trovare una collisione diventi probabile (si dice che la funzione hash offre $n/2$ bit di sicurezza). Con SHA256, con digest di 256 bit, la sicurezza contro collisioni è di circa $2^{128}$ tentativi (nota bene solo per collisioni, non per preimmagini o seconde preimmagini che invece hanno sicurezza di $2^{256}$ tentativi).

### Integrare hash per MAC

Abbiamo quindi una funzione che produce un'impronta e rende difficile trovare messaggi diversi con la stessa impronta.  
Un esempio concreto di applicazione è **sigillare digitalmente un database**: supponiamo che $D$ contenga i voti di un'elezione elettronica. Alla fine della giornata calcolo il digest $d = H(D)$ e lo conservo in un luogo protetto, ad esempio dandolo a un notaio. Il giorno dopo calcolo l'hash del database disponibile, $D'$, e verifico se $H(D') = d$. Se sì, il database non è stato alterato. Se no, qualcuno ha modificato i dati (chiaramente è necessario però che il digest $d$ sia stato protetto, altrimenti l'attaccante potrebbe modificare entrambi). 

Se invece trasmettessi sulla rete $M$ e $H(M)$ allora l'attaccante può sostituire il messaggio $M'$ e calcolare autonomamente $H(M')$, dato che la funzione hash è pubblica e non richiede una chiave.  
Per ottenere un **MAC (Message Authentication Code)** è necessario che la generazione del tag dipenda anche da una **chiave segreta condivisa $T = \text{MAC}(K, M)$**. In questo modo il destinatario che conosce $K$ può verificare il tag e l'attaccante deve avere difficoltà a costruire un tag valido per un messaggio nuovo.

Quindi la nuova domanda è: **come inseriamo correttamente la chiave in una costruzione basata su hash?**

Le proposte più immediate sono $T = H(K || M)$ oppure $T = H(M || K)$ o ancora $T = H(K || M || K)$, l'intuizione è comprensibile in quanto l'attaccante non conosce $K$ e quindi sembra non possa calcolare il tag.

Tuttavia le proprietà di una funzione hash crittografica non garantiscono automaticamente la sicurezza di queste costruzioni come MAC.  
Ad esempio la resistenza alle collisioni riguarda la difficoltà di trovare due input con la stessa impronta, mentre la sicurezza MAC riguarda invece la difficoltà di produrre una nuova coppia messaggio-tag valida, dopo aver eventualmente osservato altri tag. **Questi sono obiettivi ben distinti**.

Per hash con particolari strutture interne come SHA256, una concatenazione semplice di chiave e messaggio può essere vulnerabile ad attacchi che non richiedono di trovare collisioni né di recuperare la chiave. Vedremo tra poco la spiegazione di questi attacchi.

Tornando ora al concetto di hash ideale, questa sarebbe un **random oracle**. Possiamo immaginarlo come un modello che tiene un registro per cui quando riceve un input mai visto, estrae u.a.r. un digest casuale di $n$ bit e memorizza la coppia input-digest, e quando riceve nuovamente lo stesso input restituisce quello già memorizzato. Ciò garantirebbe la ripetibilità della funzione hash.  
Nel modello Random Oracle, conoscere $H(K||M)$ non aiuta a calcolare $H(K||M')$ per un messaggio diverso $M'$, perché il digest è casuale e non c'è alcuna struttura da sfruttare. In tal caso quindi non ci porremmo il problema.

Tuttavia nella realtà implementare un random oracle è impraticabile e ci si basa su algoritmi concreti con una struttura interna ben specifica che simula il comportamento di un random oracle, soddisfando ottime proprietà di sicurezza ma non comportandosi davvero come un random oracle e quindi con potenziali vulnerabilità.

#### Come funziona internamente SHA256
Abbiamo descritto l'hash come una funzione che dato un msg $M$ di lunghezza variabile produce un digest $H(M)$ di lunghezza fissa. Vediamo ora nel caso specifico di SHA256 come avviene il calcolo.

La funzione SHA256 è basata su un'architettura di tipo **Merkle-Damgård**. In questa architettura, il messaggio $M$ viene prima completato con il padding e diviso in blocchi da 512 bit $B_1, B_2, \ldots, B_r$.  
Ogni blocco viene poi processato insieme a uno stato interno di 256 bit attraverso una funzione di compressione $F$: $h_i = F(h_{i-1}, B_i)$, dove $h_0$ è un valore iniziale fissato.

Quindi ogni applicazione di $F$ prende 256 + 512 = 768 bit di input e produce un nuovo stato di 256 bit. Lo stato finale è proprio il digest: $H(M) = h_r$.  
Un'analogia utile è quella di mescolare ingredienti: aggiungi una porzione, mescoli, aggiungi un'altra porzione, mescoli di nuovo, dove lo stato corrente riassume quanto elaborato fino a quel momento.

Per iniziare serve uno stato iniziale $h_0 = IV$, e IV è una costante pubblica fissata dall'algoritmo. Non è da confondere con gli IV o nonce delle cifrature, che possono dover cambiare ad ogni messaggio. Qui infatti per ogni messaggio vogliamo ottenere sempre lo stesso digest.+

Per quanto riguarda il padding, al messaggio SHA256 aggiunge un bit '1', un opportuno numero di bit '0' e infine la lunghezza del messaggio originale in bit, rappresentata su 64 bit. Indicando con $l$ la lunghezza del messaggio originale in bit e $z$ il numero di bit '0' da aggiungere, il padding è calcolato in modo che:
$$l + 1 + z + 64 \equiv 0 \mod 512$$
ossia la somma deve essere un multiplo di 512.  
La lunghezza serve a rendere la codifica del messaggio non ambigua e fa parte del cosiddetto **Merkle-Damgård strengthening**. Quindi per SHA256 lo standard considera messaggi di lunghezza inferiore a $2^{64}$ bit.

Con le opportune condizioni e questo padding, la resistenza alle collisioni della funzione di compressione $F$ implica la resistenza alle collisioni della costruzione iterata. Ciò non dimostra però che qualsiasi MAC costruito aggiungendo una chiave sia sicuro, come anticipato prima. 

Un vantaggio pratico di SHA256 è l'elaborazione in **streaming**: si può leggere e processare un blocco alla volta senza caricarlo interamente in memoria, conservando solo lo stato corrente, un piccolo buffer e la lunghezza elaborata.

#### Chiave alla fine: $T = H(M || K)$
La prima proposta di MAC consiste nel concatenare la chiave segreta dopo il messaggio. Ci sono due problemi:
1. **L'attaccante può accelerare la ricerca esaustiva della chiave**: supponiamo che l'attaccante conosca $M$ e il relativo tag $T = H(M || K)$.   Allora può provare tutte le possibili chiavi $K'$ e calcolare $H(M || K')$ fino a trovare una corrispondenza con $T$.  
Il punto è che nel fare ciò l'attaccante può sfruttare il fatto che SHA256 lavora dividendo il messaggio in blocchi da 512 bit e calcolando lo stato interno $h_i$ a ogni blocco iterativamente. Dato che $M$ è noto allora l'attaccante può elaborare i blocchi che lo contengono una sola volta, conservare lo stato ottenuto e ricalcolare solo la parte finale per ogni chiave candidata $K'$. Se il messaggio è molto lungo, questo può ridurre drasticamente il numero di operazioni necessarie per testare ogni chiave candidata.  
Attenzione al fatto che ciò **non riduce il numero di chiavi possibili, ma solo il costo di ogni tentativo**. Sarebbe a dire che l'attaccante può testare più chiavi in meno tempo, quindi la sicurezza complessiva del MAC è ridotta.
2. **Alcune collisioni possono propagarsi fino al tag**: il mittente invia come sappiamo $(M,T) \quad T = H(M || K)$. Se l'attaccante, vedendo $M$ e $T$ ma non conoscendo $K$, riesce a costruire un messaggio diverso
    $$H(M' || K) = H(M || K) = T$$
    allora può inviare $(M', T)$ al destinatario, che calcolerà $H(M' || K)$ e lo confronterà con $T$ riconoscendo il messaggio valido! Si creerebbe un problema di **integrità e autenticazione**.  
    Il punto è che l'attaccante può fare ciò se riesce a trovare un messaggio $M'$ tale che lo stato interno prodotto da SHA256 dopo aver elaborato $M'$ sia lo stesso che quello prodotto da $M$. In tal caso, dato che la chiave $K$ è la stessa, il digest finale sarà lo stesso e viene quindi prodotto lo stesso tag.   
    In pratica l'attaccante è facilitato perché quindi può fare quanto segue (ipotizzando possa chiedere tag di messaggi scelti da lui, secondo il modello di sicurezza MAC per cui deve valere **unforgeability**): cerca due messaggi $M_1$ e $M_2$ distinti che producano lo stesso stato interno prima dell'elaborazione di $K$, poi chiede al sistema di autenticare solo $M_1$. Allora il sistema calcola automaticamente $T_1 = H(M_1 || K)$ e lo lascia all'attaccante.  
    Allora l'attaccante può inviare $(M_2, T_1)$ e il messaggio $M_2$ sarà comunque accettato come valido.

#### Chiave all'inizio: $T = H(K || M)$
Mettendo la chiave all'inizio l'attaccante non può più precomputare la parte iniziale senza conoscere la chiave. Tuttavia compare un problema più grave: il **length extension attack** aka **expansion attack**.

Il digest pubblicato con SHA256 è anche lo stato finale dell'elaborazione iterata. Il mittente calcola il tag come $T = H(K || M)$, e quindi SHA256 nel processo interno aggiunge automaticamente un padding $P$ ed elabora a blocchi la concatenazione $K || M || P$, aggiornando iterativamente lo stato interno e restituendo il digest finale $T$.  
Il mittente invierà $(M, T)$ al destinatario, per cui l'attaccante man in the middle conoscerà $M$ e $T$, ma non $K$.

A questo punto **l'attaccante può continuare il calcolo dello stato interno a partire da $T$**, in quanto ogni passo in SHA256 dipende solo dallo stato interno e dal blocco successivo.  
In particolare sceglie dei nuovi dati aggiuntivi $E$ e usa una procedura che continua SHA256 a partire dallo stato $T$. Non si tratta quindi del normale calcolo $H(T||E)$: imposta $T$ come stato interno iniziale invece di partire dall'IV standard.

<img src="imgs/shaa.png" width="300">

Quindi l'attaccante invierà il nuovo messaggio più lungo $M^* = M || P || E$ e il nuovo tag $T^*$, ossia il nuovo tag calcolato continuando SHA256 dallo stato $T$ concatenato con E. Il vecchio padding $P$ è incluso nel messaggio inviato e per costruirlo l'attaccante deve indovinare solo la **lunghezza di $K$**, non il suo contenuto.

A questo punto il destinatario riceve $(M^*, T^*)$ e calcola $H(K || M^*)$. Ma $M^* = M || P || E$, quindi il destinatario calcola $H(K || M || P || E)$ ossia SHA256 elabora $K || M || P || E || P^*$.  
Il calcolo ha due tratti: $K || M || P$ produce lo stato interno $T$, e poi $E || P^*$ produce lo stato interno finale $T^*$, che è proprio il tag inviato dall'attaccante. Pertanto il destinatario accetta il messaggio come valido, anche se l'attaccante non conosce la chiave $K$, violando quindi la sicurezza del MAC.

Nota bene quindi che l'attacco non produce il tag del semplice messaggio $M || E$, ma del messaggio più lungo $M || P || E$.  
Il nuovo messaggio potrebbe apparire strano, in quanto contiene byte del padding. Per definizione crittografica è comunque una falsificazione, e perché diventi un attacco utile contro un'applicazione bisognerebbe vedere come l'applicazione interpreta quei byte e l'estensione.

**In tutto ciò non abbiamo trovato mai una collisione (chiaro, è computazionalmente difficile) e non abbiamo recuperato la chiave, abbiamo solo sfruttato la struttura iterativa dell'hash.**

### HMAC
La soluzione all'integrazione di hash e chiave è l'algoritmo **HMAC** (Hash-based Message Authentication Code; Blare, Canetti e Krawczyk, 1996), una costruzione progettata specificatamente per ottenere un MAC da una funzione hash.

Tre vantaggi:
1. usa un hash esistente, senza richiedere di modificarne l'algoritmo
2. ha un'analisi di sicurezza rigorosa, basata su precise assunzioni sulla funzione hash sottostante
3. aggiunge un costo contenuto rispetto al calcolo dell'hash del messaggio.

Intuitivamente l'idea è **calcolare un hash interno con la chiave, poi proteggere il risultato con un secondo hash che usa ancora materiale segreto**.

La formula è la seguente:
$$\text{HMAC}_K(M) = H((K^+ \oplus opad) || H((K^+ \oplus ipad) || M))$$
la possiamo scomporre in due passaggi:
$$I = H((K^+ \oplus ipad) || M) \quad \text{e} \quad T = H((K^+ \oplus opad) || I)$$
dove $I$ è il risultato interno mentre $T$ è il tag HMAC trasmesso.

$K^+$ rappresenta invece la chiave normalizzata alla dimensione del blocco dell'hash (ricordiamo che per SHA256 il blocco è di 512 bit, quindi $K^+$ è lungo 64 byte). Se la chiave originale $K$ è più corta del blocco, si aggiungono byte 0. Se più lunga, prima si calcola il suo hash e poi si completa il risultato con zeri.  
Aggiungere zeri ovviamente **non aumenta l'entropia della chiave** ma serve solo a garantire la lunghezza corretta per l'operazione di XOR con i pad.

$ipad$ e $opad$ sono due costanti fisse e pubbliche, ripetute fino alla dimensione del blocco:
$$ipad = 0x36 \cdot 64 \text{ byte ripetuti}, \quad opad = 0x5c \cdot 64 \text{ byte ripetuti}.$$
Gli XOR producono due blocchi diversi:
$$K_{\text{in}} = K^+ \oplus ipad \quad \text{e} \quad K_{\text{out}} = K^+ \oplus opad$$
Sono entrambi derivati dalla stessa chiave ergo non sono chiavi indipendenti.

<img src="imgs/hmac.png" width="500">

#### Perché HMAC blocca l'extension attack?
La parte interna somiglia al MAC vulnerabile con chiave come prefisso $I = H(K_{\text{in}}||M)$. Se infatti pubblicassimo direttamente $I$, avremmo di nuovo il problema dell'estensione.  

Tuttavia HMAC pubblica $T = H(K_{\text{out}}||I)$, è come se stessimo "sigillando" il risultato interno. Infatti l'hash esterno non è da vedere come una cifratura, ma come qualcosa che impedisce all'attaccante di avere direttamente a disposizione lo stato interno $I$ da cui estenedere il messaggio.  

In pratica infatti se l'attaccante provasse a modificare $M$, dovrebbe ottenere un nuovo risultato interno $I'$ per poi produrre un nuovo tag $T' = H(K_{\text{out}}||I')$, ma non può farlo semplicemente continuando il calcolo del tag pubblicato $T = H(K_{\text{out}}||I)$.

(infatti anche se conoscesse $I = H(K_{\text{in}}||M)$, potrebbe calcolare un nuovo digest interno $I' = H(K_{\text{in}}||M')$. Tuttavia ciò sarebbe comunque inutile perché il tag valido di quel nuovo messaggio sarebbe $T' = H(K_{\text{out}}||I')$, e l'attaccante non può calcolare $T'$ senza conoscere la chiave $K_{\text{out}}$).

Inoltre $I$, in quanto hash, è una stringa di lunghezza fissa. Quindi se l'attaccante tentasse una length extension direttamente sul digest esterno $T$, il risultato corrisponderebbe a una stringa del tipo $K_{\text{out}} || I || padding || E$, ma **il verificatore HMAC inserisce dopo K_{\text{out}} solo il digest interno di lunghezza fissa**, per cui il verificatore non accetterebbe il messaggio esteso.

(in generale inoltre lo stato iniziale da cui il primo blocco interno dipende non è solo l'IV standard, ma qualcosa che dipende anche dalla chiave segreta $s_in = F(IV, K_{\text{in}})$, quindi l'attaccante non può calcolare lo stato interno iniziale senza conoscere la chiave).

Quanto alle prestazioni: il messaggio lungo $M$ viene processato una sola volta, mentre il ramo esterno elabora un digest di lunghezza fissa, corto, per cui **il costo aggiuntivo è sostanzialmente costante**.

### Il concetto di prova di sicurezza
Una prova di sicurezza è una dimostrazione matematica per cui:
$$\text{se valgono certe assunzioni (es. hash crittografica)} \Rightarrow \text{la costruzione soddisfa la proprietà di sicurezza desiderata}$$
Una prova di sicurezza non garantisce incondizionatamente che non esista alcun attacco, ma spiega come un certo tipo di attacco alla costruzione si collegherebbe a una violazione di proprietà della primitiva sottostante (es. se un attaccante viola HMAC, allora viola la resistenza alle collisioni della funzione hash sottostante).

Inoltre $A \Rightarrow B$ non implica $\neg A \Rightarrow \neg B$. Ad esempio per MD5 l'esser venuti meno alla resistenza delle collisioni non implica automaticamente che HMAC-MD5 sia falsificabile (qui A = "MD5 soddisfa le assunzioni di sicurezza", B = "HMAC costruito con quell'hash è sicuro").

Infatti esistono risultati successivi a Bellare che affermano proprio come la resistenza alle collisioni dell'hash non sia una condizione necessaria per alcune prove di sicurezza di HMAC.

In conclusione, bisogna usare una costruzione progettata per l'obiettivo preciso:
- Calcolare un'impronta di un contenuto --> hash crittografica
- Autenticare un messaggio con una chiave segreta --> MAC, ad esempio HMAC
- Nascondere un contenuto --> cifratura (qui non va bene usare hash perché non è invertibile, serve un modo per decifrare!)
- Impedire il riutilizzo di un messaggio autentico --> meccanismo di freshness, ad esempio nonce o timestamp.

<img src="imgs/summary.png" width="400">